# EFT-SP — VMamba Small: diagnóstico compartilhado

Adaptação experimental baseada na célula Tiny que passou no Runpod. Último estágio + norm pré-treinada; entrada 256, features 8×8 interpoladas para 32×32. Forward e alinhamento oficiais herdados.

São 12 atualizações, seis episódios duas vezes, batch 1, float32, AdamW (lr=1e-5, weight_decay=0), CE + alinhamento. Todos os parâmetros habilitados apenas para diagnóstico. Pesos restaurados no final. SAM e máscaras manuais não participam.

**Execute uma variante por vez. Encerre o kernel anterior via Shut Down Kernel antes de abrir a próxima.** Para Tiny, este notebook é a versão compartilhada para uso futuro; não precisa repetir o teste agora.

Os seis episódios não constituem validação independente. LoRA/full FT, orçamento, augmentations e divisão treino/val ainda serão definidos. Não há treinamento longo nem checkpoint para retomada neste diagnóstico.


In [ ]:
from pathlib import Path
import sys

REPO = Path("/workspace/ProtoSAM-repro")
PASTA = REPO / "exp/eft_sp_diagnostico_v1"
assert (PASTA / "eft_vmamba_compartilhado.py").is_file(), PASTA
for caminho in (REPO, PASTA):
    if str(caminho) not in sys.path:
        sys.path.insert(0, str(caminho))

from eft_vmamba_compartilhado import construir_modelo, testar_otimizacao


In [ ]:
assert "modelo_eft" not in globals(), "O modelo já está carregado. Use a próxima célula."
modelo_eft = construir_modelo("small", REPO)
print("Encoder:", modelo_eft.variante_eft)
print("Parâmetros do encoder:", sum(p.numel() for p in modelo_eft.encoder.parameters()))


In [ ]:
historico, pasta_registros = testar_otimizacao(modelo_eft)
print("Concluído:", pasta_registros)


In [ ]:
# Opcional, após um teste bem-sucedido. Encerre o kernel ao trocar de notebook.
import gc
import torch
if "modelo_eft" in globals():
    del modelo_eft
gc.collect()
torch.cuda.empty_cache()
